# RK1828 Depth Anything V3 Multi-view Depth Demo


[Depth Anything V3](https://github.com/ByteDance-Seed/depth-anything-3)
(ByteDance) multi-view depth estimation. The model is split into three
sub-models (local / global / head) with a fixed input of **10 views**, producing
one depth heatmap per view. About 12 s for all 10 views on the RK1828.

| Step | Content |
|---|---|
| 1 | Download models and the 10 sample frames (from huggingface.co) |
| 2 | Extract the prebuilt demo |
| 3 | Build the image list and run |
| 4 | View the depth maps |

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("depth-anything-v3-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/depth_anything_v3/demo_x86_64.tar.gz"),
    ("depth-anything-v3-rknn3-rk1828", "model/da3_base_global.rknn", "demos/depth_anything_v3/model/da3_base_global.rknn"),
    ("depth-anything-v3-rknn3-rk1828", "model/da3_base_global.weight", "demos/depth_anything_v3/model/da3_base_global.weight"),
    ("depth-anything-v3-rknn3-rk1828", "model/da3_base_head.rknn", "demos/depth_anything_v3/model/da3_base_head.rknn"),
    ("depth-anything-v3-rknn3-rk1828", "model/da3_base_head.weight", "demos/depth_anything_v3/model/da3_base_head.weight"),
    ("depth-anything-v3-rknn3-rk1828", "model/da3_base_local.rknn", "demos/depth_anything_v3/model/da3_base_local.rknn"),
    ("depth-anything-v3-rknn3-rk1828", "model/da3_base_local.weight", "demos/depth_anything_v3/model/da3_base_local.weight"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_01.jpg", "demos/depth_anything_v3/model/frame_01.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_02.jpg", "demos/depth_anything_v3/model/frame_02.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_03.jpg", "demos/depth_anything_v3/model/frame_03.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_04.jpg", "demos/depth_anything_v3/model/frame_04.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_05.jpg", "demos/depth_anything_v3/model/frame_05.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_06.jpg", "demos/depth_anything_v3/model/frame_06.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_07.jpg", "demos/depth_anything_v3/model/frame_07.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_08.jpg", "demos/depth_anything_v3/model/frame_08.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_09.jpg", "demos/depth_anything_v3/model/frame_09.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_10.jpg", "demos/depth_anything_v3/model/frame_10.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/images.txt", "demos/depth_anything_v3/model/images.txt"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Extract the prebuilt demo

`demo_x86_64.tar.gz` contains the official x86_64 C++ demo and its runtime
libraries (`lib/`). Hugging Face does not host raw executables, so they are
shipped as a tarball; extract once (re-running skips automatically).

> **Why a C++ demo instead of pure Python?**
> Notebooks 01-05 (classification / detection / FastSAM / pose / segmentation)
> are a single model with one forward pass and simple post-processing, so they
> walk through everything in Python (`rknn3.py`). This demo's pipeline is far
> more complex — chained models, tokenizer, KV cache, sliding-window /
> streaming state, hundreds of lines of orchestration — reimplementing it in
> Python would be costly and risk diverging from the official behavior.
> So we run the official `rknn3-model-zoo` C++ demo directly; source code:
> [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> under `examples/`.

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/depth_anything_v3/demo_x86_64.tar.gz", "demos/depth_anything_v3")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. Build the 10-view image list and run

The repo ships 10 sample frames of the same scene (`model/frame_01.jpg` ~
`frame_10.jpg`); the demo wants a list file with exactly 10 image paths.

In [ ]:
import glob, os

DEMO = "demos/depth_anything_v3"
frames = sorted(glob.glob(os.path.join(DEMO, "model/frame_*.jpg")))
assert len(frames) == 10, f"need exactly 10 frames, got {len(frames)}"
with open(os.path.join(DEMO, "image_list.txt"), "w") as f:
    f.write("\n".join(os.path.abspath(p) for p in frames))  # demo needs absolute paths
show_images(frames[:5], max_width=320)
print(f"{len(frames)} frames, list written to image_list.txt")

In [ ]:
run_demo(["./rknn_da3_base_demo",
          "model/da3_base_local.rknn", "model/da3_base_global.rknn",
          "model/da3_base_head.rknn",
          "image_list.txt", "10", "depth_vis"], DEMO)

## 4. View the depth heatmaps (brighter = closer)

In [ ]:
import glob

depth_imgs = sorted(glob.glob(os.path.join(DEMO, "depth_vis/*")))
print(f"{len(depth_imgs)} depth maps generated")
show_images(depth_imgs[:5], max_width=320)

## Summary

- Splitting a big model into sub-models, converted and pipelined separately,
  is the standard RKNN approach for very large models.
- With your own photos: exactly 10 views, ideally the same scene from
  different angles.